# 01 · EDA — classification & detection datasets
### *From Diagnosis to Decision* — ICA 2026

Exploratory analysis of the datasets that feed the **diagnosis** stage:
PlantVillage (lab training source), PlantWild & PlantDoc (in-the-wild eval),
FieldPlant (pathologist-supervised field), Cassava (viral classes).

For every present dataset this notebook reports: image count, class count,
**class-imbalance ratio**, resolution & aspect-ratio distributions, corrupt/
unreadable files, and a sample montage. A cross-dataset summary quantifies the
**lab→field domain gap** that motivates the paper.

> The notebook **auto-detects** which datasets are downloaded (run `00` first)
> and analyses whatever is present. Locally it will typically find only
> **PlantDoc**; on Colab it analyses all of them.

## 1 · Setup

In [ ]:
# --- Environment config: works on Google Colab AND locally --------------------
import os, sys, pathlib

def in_colab():
    return "google.colab" in sys.modules or os.path.exists("/content")

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/diagnosis-to-decision")
else:
    # local fallback: repo root (edit if you cloned elsewhere)
    PROJECT_ROOT = pathlib.Path(
        os.environ.get("ICA_PROJECT_ROOT", pathlib.Path.cwd().parents[0])
    )

DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_MAPPING = PROJECT_ROOT / "data" / "mapping"
FIGDIR       = PROJECT_ROOT / "reports" / "figures"
for p in (DATA_RAW, DATA_INTERIM, DATA_MAPPING, FIGDIR):
    p.mkdir(parents=True, exist_ok=True)

print("Colab:", in_colab())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW exists:", DATA_RAW.exists())

In [ ]:
import warnings, io
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = False   # we WANT to detect truncation
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}

In [ ]:
# --- Which datasets are actually downloaded? Notebooks adapt to what's present.
import pandas as pd

CANDIDATES = {
    "plantvillage": DATA_RAW / "plantvillage",
    "plantwild":    DATA_RAW / "plantwild",
    "plantseg":     DATA_RAW / "plantseg",
    "plantdoc":     DATA_RAW / "plantdoc",
    "fieldplant":   DATA_RAW / "fieldplant",
    "cassava":      DATA_RAW / "cassava",
    "master":       DATA_RAW / "master_plant_disease",
    "bracol":       DATA_RAW / "bracol",
    "rocole":       DATA_RAW / "rocole",
}
PRESENT = {k: v for k, v in CANDIDATES.items() if v.exists() and any(v.rglob("*"))}
print("Present datasets:", list(PRESENT) or "(none yet — run 00_download_verify first)")

## 2 · Generic scanners

Two loaders cover every source: an **ImageFolder** scanner
(`<split>/<class>/<img>` — PlantDoc, FieldPlant folder export, Cassava-by-folder)
and a **Hugging Face arrow** scanner (PlantVillage, PlantWild cached by `00`).
Each returns a tidy row-per-image DataFrame with `path, label, split`.

In [ ]:
def scan_imagefolder(root):
    """root/<split>/<class>/<img> OR root/<class>/<img>. Returns DataFrame."""
    root = pathlib.Path(root)
    rows = []
    splits = [d for d in root.iterdir() if d.is_dir() and d.name.lower() in
              {"train","test","val","valid","validation"}]
    if splits:
        for sp in splits:
            for cls in sorted(p for p in sp.iterdir() if p.is_dir()):
                for img in cls.iterdir():
                    if img.suffix.lower() in IMG_EXTS:
                        rows.append((str(img), cls.name, sp.name))
    else:  # flat class dirs
        for cls in sorted(p for p in root.iterdir() if p.is_dir()):
            for img in cls.rglob("*"):
                if img.suffix.lower() in IMG_EXTS:
                    rows.append((str(img), cls.name, "all"))
    return pd.DataFrame(rows, columns=["path","label","split"])

def scan_hf_arrow(root):
    """Loads a datasets.save_to_disk() dump; returns label-only frame (paths lazy)."""
    from datasets import load_from_disk
    ds = load_from_disk(str(pathlib.Path(root) / "hf_arrow"))
    frames = []
    for split in ds:
        d = ds[split]
        labelnames = d.features["label"].names if hasattr(d.features["label"],"names") else None
        labs = d["label"]
        labs = [labelnames[i] for i in labs] if labelnames else labs
        frames.append(pd.DataFrame({"path": [f"{split}:{i}" for i in range(len(d))],
                                    "label": labs, "split": split}))
    return pd.concat(frames, ignore_index=True), ds

def load_dataset_frame(name, path):
    if (pathlib.Path(path) / "hf_arrow").exists():
        df, _ = scan_hf_arrow(path); return df
    return scan_imagefolder(path)

## 3 · Per-dataset EDA

`analyse()` computes counts, the imbalance ratio (largest class ÷ smallest),
and — by sampling up to `n_probe` images — resolution/aspect stats and a
corrupt-file count. Sampling keeps it fast on Colab; set `n_probe=None` for a
full scan.

In [ ]:
def probe_images(paths, n_probe=400, seed=0):
    """Open a sample of images; collect (w,h) and count corrupt/truncated."""
    rng = np.random.default_rng(seed)
    real = [p for p in paths if ":" not in str(p)]        # skip lazy HF refs
    if not real:
        return pd.DataFrame(columns=["w","h"]), 0, 0
    sample = real if (n_probe is None or len(real)<=n_probe) else \
             list(rng.choice(real, n_probe, replace=False))
    dims, corrupt = [], 0
    for p in sample:
        try:
            with Image.open(p) as im:
                im.verify()                                # catch truncation
            with Image.open(p) as im:
                dims.append(im.size)                       # (w,h)
        except Exception:
            corrupt += 1
    d = pd.DataFrame(dims, columns=["w","h"])
    return d, corrupt, len(sample)

def analyse(name, df, n_probe=400):
    counts = df["label"].value_counts()
    dims, corrupt, probed = probe_images(df["path"].tolist(), n_probe)
    imb = (counts.max()/counts.min()) if len(counts)>1 else 1.0
    summary = dict(
        dataset=name, n_images=len(df), n_classes=df["label"].nunique(),
        imbalance_ratio=round(float(imb),1),
        largest_class=f"{counts.idxmax()} ({counts.max()})",
        smallest_class=f"{counts.idxmin()} ({counts.min()})",
        splits="|".join(sorted(df["split"].unique())),
        corrupt_in_sample=corrupt, probed=probed,
        median_w=int(dims["w"].median()) if len(dims) else None,
        median_h=int(dims["h"].median()) if len(dims) else None,
        median_megapix=round(float((dims["w"]*dims["h"]).median()/1e6),2) if len(dims) else None,
    )
    return summary, counts, dims

In [ ]:
# Run EDA over every present dataset
summaries, per_ds = [], {}
for name, path in PRESENT.items():
    try:
        df = load_dataset_frame(name, path)
        s, counts, dims = analyse(name, df)
        summaries.append(s); per_ds[name] = (df, counts, dims)
        print(f"[ok] {name}: {s['n_images']} imgs, {s['n_classes']} classes, "
              f"imbalance {s['imbalance_ratio']}x, corrupt {s['corrupt_in_sample']}/{s['probed']}")
    except Exception as e:
        print(f"[skip] {name}: {type(e).__name__}: {e}")

summary_df = pd.DataFrame(summaries)
if len(summary_df):
    summary_df.to_csv(DATA_INTERIM / "eda_summary.csv", index=False)
summary_df

## 4 · Class-distribution & resolution figures

In [ ]:
def plot_dataset(name, counts, dims):
    fig, ax = plt.subplots(1, 2, figsize=(13,4))
    top = counts.sort_values(ascending=False).head(30)
    ax[0].bar(range(len(top)), top.values, color="#4C78A8")
    ax[0].set_title(f"{name} — class distribution (top {len(top)})")
    ax[0].set_xlabel("class (rank)"); ax[0].set_ylabel("images")
    ax[0].axhline(counts.mean(), ls="--", c="grey", lw=1, label="mean")
    ax[0].legend()
    if len(dims):
        ax[1].scatter(dims["w"], dims["h"], s=6, alpha=0.3, color="#E45756")
        ax[1].set_title(f"{name} — resolution (sampled)")
        ax[1].set_xlabel("width px"); ax[1].set_ylabel("height px")
    else:
        ax[1].text(0.5,0.5,"resolution not probed\n(HF lazy refs)",ha="center")
    plt.tight_layout()
    fig.savefig(FIGDIR / f"eda_{name}.png", bbox_inches="tight")
    plt.show()

for name,(df,counts,dims) in per_ds.items():
    plot_dataset(name, counts, dims)

## 5 · Sample montage (visual sanity check)

In [ ]:
def montage(name, df, k=12, seed=1):
    real = df[~df["path"].astype(str).str.contains(":")]
    if not len(real):
        print(f"{name}: images are lazy HF refs — load via datasets to view."); return
    s = real.sample(min(k,len(real)), random_state=seed)
    cols=4; rows=int(np.ceil(len(s)/cols))
    fig,axes=plt.subplots(rows,cols,figsize=(12,3*rows))
    for ax,(_,r) in zip(np.array(axes).ravel(), s.iterrows()):
        try:
            ax.imshow(Image.open(r["path"]).convert("RGB"));
        except Exception as e:
            ax.text(0.5,0.5,"unreadable",ha="center")
        ax.set_title(r["label"][:22], fontsize=8); ax.axis("off")
    for ax in np.array(axes).ravel()[len(s):]: ax.axis("off")
    fig.suptitle(f"{name} — random samples"); plt.tight_layout(); plt.show()

for name,(df,_,_) in per_ds.items():
    montage(name, df)

## 6 · Cross-dataset domain gap

The paper's central claim is that lab-trained accuracy does **not** transfer to
the field. This table juxtaposes the *conditions* and *scale* of each set — the
qualitative half of that argument. The quantitative half (accuracy drop) is
produced in `05_baseline_pretraining.ipynb`.

In [ ]:
CONDITIONS = {
    "plantvillage":"laboratory, uniform background",
    "plantwild":   "in-the-wild (web image search)",
    "plantdoc":    "field + some lab, web-sourced (label noise)",
    "fieldplant":  "real plantation, pathologist-supervised",
    "cassava":     "smallholder field (Uganda)",
    "master":      "merged multi-source",
    "bracol":      "controlled coffee-leaf capture",
    "rocole":      "coffee field, robusta",
}
if len(summary_df):
    t = summary_df.copy()
    t["conditions"] = t["dataset"].map(CONDITIONS)
    cols = ["dataset","conditions","n_images","n_classes","imbalance_ratio",
            "median_megapix","corrupt_in_sample"]
    display(t[cols])
    print("\nTakeaway: training on a lab set (uniform ratio ~1x, clean) and "
          "evaluating on field sets (high imbalance, label noise, varied "
          "resolution) is a genuine distribution shift — quantified in nb 05.")

---
**Notes for the paper's Data section**
- Report the **imbalance ratio** per dataset; Cassava (~CMD 62%) and any long-tail
  set need class-balanced sampling or loss reweighting — state which.
- Report **corrupt/truncated** file counts; exclude them and say so.
- PlantDoc mixes lab and field images and was annotated without a pathologist —
  disclose as a threat to validity.

**Next:** `02_severity_plantseg.ipynb`.